In [1]:
import pandas as pd
import numpy as np
import string
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("test.csv")

# Check class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())
print("\nPercentage distribution:")
print(df['sentiment'].value_counts(normalize=True) * 100)

# -----------------------------
# Data Cleaning
# -----------------------------
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['Cleaned_Text'] = df['text'].apply(clean_text)

# -----------------------------
# Feature Extraction (TF-IDF)
# -----------------------------
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['Cleaned_Text'])
y = df['sentiment'].str.lower()

# -----------------------------
# Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -----------------------------
# Hyperparameter Tuning for Logistic Regression
# -----------------------------
param_grid = {
    'C': [0.01, 0.1, 1, 10],          # regularization strength
    'solver': ['liblinear', 'lbfgs'],  # solvers
    'penalty': ['l2'],                 # regularization type
    'max_iter': [1000]                  # ensure convergence
}

lr = LogisticRegression(multi_class='ovr', random_state=42)

grid_search = GridSearchCV(lr, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

# Train best model
best_lr = grid_search.best_estimator_
y_pred = best_lr.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred, labels=y.unique())

print("\nModel Performance Metrics (Tuned Logistic Regression):")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==============================
# User Input for Prediction
# ==============================

#Load saved objects
import joblib

# Save trained Logistic Regression model
joblib.dump(best_lr, "SentimentAnalysis_logistic regression.joblib")

# Save TF-IDF vectorizer
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")

print("\nModel and vectorizer saved successfully using Joblib!")

print("\n--- Predict which sentiment category your review belongs to ---")
new_text = input("Enter your review: ")

# Clean input
cleaned_text = clean_text(new_text)

# Transform using TF-IDF
text_vector = vectorizer.transform([cleaned_text])

# Predict using trained Logistic Regression
prediction = best_lr.predict(text_vector)[0]

print("\nPrediction:", prediction.upper())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Class Distribution:
sentiment
neutral     1020
positive     774
negative     705
Name: count, dtype: int64

Percentage distribution:
sentiment
neutral     40.816327
positive    30.972389
negative    28.211285
Name: proportion, dtype: float64


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Best Parameters: {'C': 10, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}

Model Performance Metrics (Tuned Logistic Regression):
Accuracy: 0.612
Precision: 0.6279208789476796
Recall: 0.612
F1-Score: 0.6111515897435897
Confusion Matrix:
 [[144  30  30]
 [ 50  94  11]
 [ 70   3  68]]

Classification Report:
               precision    recall  f1-score   support

    negative       0.62      0.48      0.54       141
     neutral       0.55      0.71      0.62       204
    positive       0.74      0.61      0.67       155

    accuracy                           0.61       500
   macro avg       0.64      0.60      0.61       500
weighted avg       0.63      0.61      0.61       500


Model and vectorizer saved successfully using Joblib!

--- Predict which sentiment category your review belongs to ---


Enter your review:  good item



Prediction: POSITIVE
